# 1. AOI Selection — Google Earth Engine

Draw an Area of Interest (AOI) on an interactive map and export it as **GeoJSON**, **KML**, and **KMZ**.

Run the cells top to bottom:
1. Authenticate / initialize Earth Engine.
2. Display the map and draw your AOI (rectangle or polygon tool, one shape).
3. Run the export cell to write the files.
4. Verify the export.

The output files (named `AOI_NAME` below) are what `2_Sentinel_Search.ipynb` reads — run this notebook first.

📖 **New to this toolkit?** See `docs/pdf/00_Getting_Started.pdf` for account setup, or
`docs/pdf/01_AOI_Selection_Guide.pdf` for this step specifically.

## Setup — authenticate and initialize Earth Engine

Set `EE_PROJECT` to a Google Cloud project that has the Earth Engine API enabled
(see https://console.cloud.google.com — any project you've registered at
https://code.earthengine.google.com/register works). Authentication uses
`auth_mode="localhost"`, which opens your browser and completes automatically
via a local redirect — no authorization code to copy/paste. It only needs to
run once per machine; after that the credentials are cached and this cell
just initializes.

In [ ]:
import ee
import geemap

EE_PROJECT = "rosy-precinct-498822-e1"  # <-- your GEE-enabled Cloud project

try:
    ee.Initialize(project=EE_PROJECT)
except Exception:
    # auth_mode="localhost": opens your browser, you log in and click Allow,
    # and a local server on this machine catches the redirect automatically —
    # no authorization code to copy/paste. (The default flow's manual-code-entry
    # mode relies on Google's deprecated "out-of-band" OAuth flow, which now
    # fails with "Cannot authenticate: Invalid request.")
    ee.Authenticate(auth_mode="localhost")
    ee.Initialize(project=EE_PROJECT)

print("Earth Engine initialized.")

## Draw the AOI

Use the rectangle or polygon tool on the left toolbar of the map below to draw
**one** shape over your area of interest. Draw only one shape — if you draw
more than one, the export step below uses the last one drawn.

In [ ]:
Map = geemap.Map(center=[20, 0], zoom=3)
Map.add_basemap("Hybrid")
Map.add_basemap("Roadmap")
Map

Map(center=[20, 0], controls=(WidgetControl(options=['position', 'transparent_bg'], position='topright', trans…

## Export the drawn AOI

Set `AOI_NAME` and `OUTPUT_DIR`, then run the cell. It reads the last shape
drawn on the map above (`Map.user_roi`) and writes `.geojson`, `.kml`, and
`.kmz` files. `2_Sentinel_Search.ipynb` looks for these same `AOI_NAME` /
`OUTPUT_DIR` values, so keep them noted if you change them.

In [ ]:
from aoi_export import export_all

AOI_NAME = "my_aoi"
OUTPUT_DIR = "output"

geometry = Map.user_roi
if geometry is None:
    raise ValueError("No shape drawn yet — draw a rectangle or polygon on the map above, then re-run this cell.")

paths = export_all(geometry, OUTPUT_DIR, name=AOI_NAME)
for fmt, path in paths.items():
    print(f"{fmt}: {path.resolve()}")

ValueError: No shape drawn yet — draw a rectangle or polygon on the map above, then re-run this cell.

## Verify the exported AOI

Load the GeoJSON back and overlay it on a fresh map as a sanity check.

In [ ]:
check_map = geemap.Map()
check_map.add_geojson(str(paths["geojson"]), layer_name=AOI_NAME)
check_map.centerObject(geometry, zoom=10)
check_map

NameError: name 'paths' is not defined

: 